1. Import Libraries

In [21]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import(
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import(
    EarlyStopping,
    ModelCheckpoint
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

2. Load DataSet

In [22]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

3. Remove Unnecessary Column

In [23]:
df.drop("customerID", axis=1, inplace = True)

4. Handle TotalCharges

In [24]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)
df["TotalCharges"].isnull().sum()

np.int64(11)

5. Handle Missing Values

In [25]:
df.dropna(inplace = True)
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

6. Separate Features and Target

In [26]:
x = df.drop("Churn", axis = 1)
y = df["Churn"]

7. Encode Target

In [27]:
le = LabelEncoder()
y = le.fit_transform(y)


8. Indentify Features Types

In [28]:
categorical_columns = x.select_dtypes(
    include = "object"
).columns

numerical_columns = x.select_dtypes(
    include = np.number
).columns

/var/folders/83/lyk88fr96bq5vfw5v_m3j0n40000gn/T/ipykernel_58471/1075560464.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x.select_dtypes(


9. One-Hot Encode Features

In [29]:
x = pd.get_dummies(
    x,
    columns = categorical_columns,
    drop_first = True
)

10. Train-Test Split

In [30]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

11. Feature Scaling

In [31]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# 1. Baseline Model

12. Build ANN

In [32]:
input_dim = x_train.shape[1]

model = Sequential([
    Dense(
        64,
        activation = "relu",
        input_shape = (input_dim,) 
    ),
    BatchNormalization(),
    Dropout(0.3),
    Dense(
        32,
        activation="relu"
    ),
    BatchNormalization(),
    Dropout(0.2),
    Dense(
        1,
        activation="sigmoid"
    )
])

/Users/vivekkumar/Desktop/project/Customer-Churn-Prediction-Deep-Learning/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


13. Compile the Model

In [33]:
model.compile(
    optimizer = Adam(
        learning_rate = 0.001
    ),
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

14. Configure Callbacks

In [34]:
#Early Stopping

early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience = 10,
    restore_best_weights = True
)

#Checkpoint

checkpoint = ModelCheckpoint(
    "../models/best_ann_model.keras",
    monitor = "val_loss",
    save_best_only = True
)

15. Train the Model

In [35]:
history = model.fit(
    x_train, y_train,
    validation_split = 0.2,
    epochs = 100,
    batch_size = 32,
    callbacks = [early_stopping, checkpoint],
    verbose = 1
)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6940 - loss: 0.6113 - val_accuracy: 0.7964 - val_loss: 0.4754
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7676 - loss: 0.4920 - val_accuracy: 0.8053 - val_loss: 0.4301
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7796 - loss: 0.4713 - val_accuracy: 0.8036 - val_loss: 0.4190
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7858 - loss: 0.4575 - val_accuracy: 0.8053 - val_loss: 0.4171
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7871 - loss: 0.4520 - val_accuracy: 0.8071 - val_loss: 0.4153
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7880 - loss: 0.4462 - val_accuracy: 0.8018 - val_loss: 0.4150
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7838 - loss: 0.4401 - val_accuracy: 0.8107 - val_loss: 0.4108
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7907 - loss: 0.4398 - val_accu

16. Predict Probabilities

In [36]:
y_pred_prob = model.predict(x_test)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


17. Convert Probability to Class

In [37]:
y_pred = (y_pred_prob >= 0.5).astype(int)
y_pred = y_pred.flatten()

18. All Metrics

In [48]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_pred_prob
)

19. Results Table

In [49]:
results = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "ROC-AUC"
        ],
        "Value": [
            accuracy,
            precision,
            recall,
            f1,
            roc_auc
        ]
    })
results

,Metric,Value
0,Accuracy,0.786780
1,Precision,0.613497
2,Recall,0.534759
3,F1 Score,0.571429
4,ROC-AUC,0.835694


# 2. Architecture Optimization

1. Create build_model function

In [40]:
def build_model(hidden_layers):
    model = Sequential()

    model.add(Dense(
        hidden_layers[0],
        activation = "relu",
        input_shape = (x_train.shape[1],)
    ))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    for units in hidden_layers[1:]:
        model.add(Dense(units, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.2))
    
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

2. Define Architectures

In [41]:
architectures = {
    "64-32": [64, 32],
    "128-64": [128,64],
    "32-16": [32, 16],
    "64": [64]
}

3. Train Each Model

In [46]:
results = []

for name, layers in architectures.items():
    print(f"\nTraining: {name}")
    model = build_model(layers)

    history = model.fit(
        x_train, y_train,
        validation_split = 0.2,
        epochs = 100,
        batch_size = 32,
        callbacks=[early_stopping],
        verbose = 0
    )

    y_prob = model.predict(x_test, verbose=0)
    y_pred = (y_prob >= 0.5).astype(int).flatten()

    results.append({
        "Architecture": name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })


Training: 64-32


/Users/vivekkumar/Desktop/project/Customer-Churn-Prediction-Deep-Learning/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training: 128-64


/Users/vivekkumar/Desktop/project/Customer-Churn-Prediction-Deep-Learning/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training: 32-16


/Users/vivekkumar/Desktop/project/Customer-Churn-Prediction-Deep-Learning/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training: 64


/Users/vivekkumar/Desktop/project/Customer-Churn-Prediction-Deep-Learning/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4. Create Results Table

In [47]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="F1",
    ascending= False
)

,Architecture,Accuracy,Precision,Recall,F1,ROC-AUC
2,32-16,0.797441,0.646865,0.524064,0.579025,0.830470
0,64-32,0.791756,0.626959,0.534759,0.577201,0.831493
1,128-64,0.787491,0.613293,0.542781,0.575887,0.826373
3,64,0.786780,0.613497,0.534759,0.571429,0.830105


### Architecture Optimization

#### Objective
Optimized the ANN architecture by evaluating multiple hidden-layer configurations while keeping all other hyperparameters constant.

#### Architectures Evaluated
- Baseline: 64 → 32
- 128 → 64
- 32 → 16
- 64

#### Results Summary
- **32 → 16** achieved the highest Accuracy (79.74%), Precision (64.69%), and F1 Score (57.90%).
- **128 → 64** achieved the highest Recall (54.28%).
- **Baseline** achieved the highest ROC-AUC (83.57%).
- The 32 → 16 architecture provided the best overall balance between performance and model complexity.

#### Selected Architecture
**32 → 16**

#### Key Learnings
- Increasing the number of neurons does not always improve performance.
- A smaller architecture generalized better on this dataset.
- F1 Score was prioritized over Accuracy for selecting the final model.

#### Outcome
The **32 → 16** ANN architecture was selected as the optimized baseline for subsequent experiments involving optimizers, learning rates, and batch sizes.